# QTrans—UCR Wafer 严格平衡二分类正式实验

本 Notebook 使用 UCR/UEA 官方 Wafer TRAIN/TEST 分割，在官方训练集和测试集内分别以固定索引下采样为 1:1 二分类。训练侧的 194 个平衡样本再固定分为 155 个训练和 39 个验证样本；测试侧保留 1330 个平衡样本，不参与检查点选择。

四个模型共用同一输入分块器、分类头和优化配置，可训练参数差严格限制在 1% 内。Accuracy 为主指标，Macro-F1 同时报告。

## 数据集公认性与边界

Wafer 是 UCR/UEA Time Series Classification Archive 的经典二分类传感器序列基准，已被明确用于超过 3 项可核查研究，例如 [2021 年 SVM 异常检测研究](https://www.atlantis-press.com/journals/jrnal/125957120/view)、[2024 年 AutoLDT 时间序列分类](https://pmc.ncbi.nlm.nih.gov/articles/PMC11608331/)、[2025 年 KAN 时间序列研究](https://onlinelibrary.wiley.com/doi/10.1155/int/9553189)和 [2025 年孪生网络异常检测研究](https://www.mdpi.com/2227-7390/13/7/1090)。官方数据集说明见 [UCR/UEA Wafer](https://timeseriesclassification.com/description.php?Dataset=Wafer)，存档记录见 [Zenodo](https://zenodo.org/records/11198387)。

Wafer 原数据已有方法报告约 99% 准确率，存在天花板效应。因此本实验若领先，只能声称在「固定平衡子集+参数匹配微型模型」下领先，不能声称取得 UCR Wafer 全尺度 SOTA。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch

from qcs_balanced_binary import (
    BINARY_SEEDS, BinaryExperimentConfig, audit_ucr_wafer,
    experiment_progress, gradient_audit_binary, paired_comparisons,
    require_complete, run_ucr_wafer_balanced,
)

pd.set_option('display.max_columns', 100)
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'qcs_balanced_binary.py').exists():
    raise RuntimeError('请从 /root/xxx/autodl 目录打开并运行本 Notebook')
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'Wafer'
ARTIFACT_ROOT = PROJECT_DIR / 'artifacts' / 'balanced_binary_qtrans'
CONFIG = BinaryExperimentConfig()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(pd.DataFrame([{'data': str(DATA_DIR), 'artifacts': str(ARTIFACT_ROOT), 'device': str(DEVICE)}]))

## 1. 强制数据、切分、参数量和梯度审计

程序会显示官方训练/测试类别数、固定平衡子集数、四模型参数量和全参数梯度检查。任一门禁失败都会立即停止。

In [ ]:
required = [DATA_DIR / 'Wafer_TRAIN.txt', DATA_DIR / 'Wafer_TEST.txt']
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)

audit = audit_ucr_wafer(DATA_DIR, CONFIG)
display(pd.DataFrame([{'features': audit['features'], **audit['official_counts']}]))
display(audit['balanced_distribution'])
display(audit['parameter_audit'])
display(gradient_audit_binary(152, CONFIG, DEVICE))

## 2. 查看断点状态

正式实验为 `4 模型 × 5 训练种子 = 20` 个任务。每个任务保存最佳验证损失检查点、训练曲线、测试预测、混淆矩阵和协议签名。

In [ ]:
progress_before = experiment_progress(ARTIFACT_ROOT, 'ucr_wafer', BINARY_SEEDS)
display(progress_before.groupby('model')['complete'].agg(['sum', 'count']))
print(f"已完成 {int(progress_before['complete'].sum())}/{len(progress_before)} 个任务")

## 3. 运行全部正式训练

`MAX_JOBS=None` 表示跑完所有剩余任务。首次可暂改为 `1` 验证服务器环境，但论文结果必须恢复 `None` 并完成 20/20。不允许依测试分数挑种子。

In [ ]:
MAX_JOBS = None

wafer_results = run_ucr_wafer_balanced(
    data_dir=DATA_DIR,
    artifact_dir=ARTIFACT_ROOT,
    seeds=BINARY_SEEDS,
    config=CONFIG,
    balance_seed=2026,
    split_seed=4096,
    max_jobs=MAX_JOBS,
    resume=True,
    device=DEVICE,
)
print(f'当前收集 {len(wafer_results)}/20 个结果')
display(wafer_results)

## 4. 完整性门禁与论文结果

20/20 个任务完成前，下方单元会拒绝产生正式对比。统计单位是 5 个预先固定的训练种子，所有种子必须完整报告。

In [ ]:
progress_after = experiment_progress(ARTIFACT_ROOT, 'ucr_wafer', BINARY_SEEDS)
require_complete(progress_after)
RESULT_DIR = ARTIFACT_ROOT / 'ucr_wafer'
results = pd.read_csv(RESULT_DIR / 'results.csv')
summary = pd.read_csv(RESULT_DIR / 'summary.csv')
display(summary)

paired_acc = paired_comparisons(results, metric='accuracy')
paired_f1 = paired_comparisons(results, metric='macro_f1')
paired_acc.to_csv(RESULT_DIR / 'paired_accuracy.csv', index=False)
paired_f1.to_csv(RESULT_DIR / 'paired_macro_f1.csv', index=False)
display(paired_acc)
display(paired_f1)

In [ ]:
ax = summary.set_index('model')[['accuracy_mean', 'macro_f1_mean']].plot.bar(
    figsize=(9, 4), ylim=(0, 1), rot=15, grid=True
)
ax.set_ylabel('score')
ax.set_title('UCR Wafer balanced official-test subset, 5 training seeds')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'formal_accuracy_macro_f1.png', dpi=300, bbox_inches='tight')
plt.show()

## 发表解读规则

如果 QTrans 领先，正确表述是「在预注册的平衡 UCR Wafer 子集和约 7.2k 参数的匹配微型模型协议下，QTrans 的平均 Accuracy/Macro-F1 最高」。若平均值不领先，必须如实报告，不能选择性保留某次训练。